# 7. Localizing objects with concepts

NAVE has no access to labels, yet its regions often isolate objects. The paper evaluates this with
weakly supervised object localization (Section 4.3, Tables 3 and 4). Each connected region of the
explanation map gets a box, and the box with the best IoU against the ground truth counts. With K = 5,
NAVE outperformed SLIC, CRAFT and CONE-SHAP.

Two boxes per region (Appendix E):
- the **outer box** (Algorithm 2), the region's bounding box;
- the **inner box** (Algorithm 3), centred on the region's median.

The paper reports inner boxes.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "dino"         # the paper's localization runs use DINO encoders; "random" to skip downloads
gt_box = (60, 30, 470, 600)    # (xmin, ymin, xmax, ymax): an approximate box around the person

In [ ]:
def outer_box(mask):                                  # Algorithm 2
    ys, xs = np.nonzero(mask)
    return xs.min(), ys.min(), xs.max(), ys.max()

def inner_box(mask):                                  # Algorithm 3
    H, W = mask.shape
    ys, xs = np.nonzero(mask)
    cx, cy = np.median(xs), np.median(ys)
    dx = min(abs(cx - xs.min()), abs(cx - xs.max()))
    dy = min(abs(cy - ys.min()), abs(cy - ys.max()))
    return np.clip(cx - dx, 0, W), np.clip(cy - dy, 0, H), np.clip(cx + dx, 0, W), np.clip(cy + dy, 0, H)

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return ix * iy / (area(a) + area(b) - ix * iy) if ix * iy else 0.0

def best_box(exp, gt, box=inner_box, min_area=0.01):
    regions = exp.segmentation()
    sizes = np.bincount(regions.ravel())
    return max(((iou(box(regions == r), gt), box(regions == r)) for r in np.nonzero(sizes >= min_area * regions.size)[0]),
               key=lambda t: t[0])

explainer = NAVEImage(model="vits16", training=TRAINING, k=5)
exp = explainer.explain(image)
for name, box in (("inner", inner_box), ("outer", outer_box)):
    score, b = best_box(exp, gt_box, box)
    print(f"{name} box: IoU {score:.2f} ({'correct' if score > 0.5 else 'missed'} at IoU > 0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
show(ax, explainer, exp)
for name, box, colour in (("inner", inner_box, "cyan"), ("outer", outer_box, "yellow")):
    _, (x0, y0, x1, y1) = best_box(exp, gt_box, box)
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=colour, lw=2, label=f"{name} box"))
x0, y0, x1, y1 = gt_box
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="lime", lw=2, ls="--", label="ground truth"))
ax.legend(loc="lower left");

### On a dataset (e.g. VOC07)

The Correct Localization metric (AP@50) counts the images whose best box has an IoU above 0.5 with
at least one ground-truth box:

```python
hits = 0
for img, gt_boxes in dataset:              # your loader: image and its list of boxes
    exp = explainer.explain(img)
    hits += any(best_box(exp, gt)[0] > 0.5 for gt in gt_boxes)
print("AP@50:", hits / len(dataset))
```

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```

We thank Ulf Brefeld (Leuphana University Lüneburg) for lending us the computing resources without which this work would not have been possible.